---
#### Structured output

Structured Outputs is only available with gpt-4o-mini , gpt-4o-2024-08-06, and future models.

---

In [5]:
import json, os
from textwrap import dedent
from openai import OpenAI

In [6]:
openai_api_key = os.getenv("OPENAI_API_KEY")
                           
client = OpenAI(
    #api_key = openai_api_key
)

In [7]:
MODEL = "gpt-4o-2024-08-06"

#### 1. Structured Output via JSON Schema in API Calls

**Example 1: Math tutor**
In this example, we want to build a math tutoring tool that outputs steps to solving a math problem as an array of structured objects.

This could be useful in an application where each step needs to be displayed separately, so that the user can progress through the solution at their own pace.

In [10]:
math_tutor_prompt = '''
    You are a helpful math tutor. 
    You will be provided with a math problem,
    and your goal will be to output a step by step solution, along with a final answer.
    For each step, just provide the output as an equation. use the explanation field to 
    detail the reasoning.
'''

- In the new GPT-4 API, `response_format` allows you to specify the format you expect for the model’s output.
- By defining a `JSON schema`, you can guide the model to generate responses in a structured format, which is particularly useful when working with data that needs to be in a predictable structure—like a step-by-step solution for math problems.

In [11]:
def get_math_solution(question):
    response = client.chat.completions.create(
        model    = MODEL,
        messages = [
            {
                "role": "system", 
                "content": dedent(math_tutor_prompt)
            },
            {
                "role": "user", 
                "content": question
            }
        ],
        response_format = {
            "type": "json_schema",          # type of response format. For structured responses
            "json_schema": {                # schema that the model should follow in its response
                "name": "math_reasoning",   # name for the schema
                "schema": {                 # core structure of the expected JSON response
                    "type": "object",       # data type at the root level
                    "properties": {         # A dictionary of expected fields in the output
                        "steps": {          # array of objects
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "explanation": {"type": "string"},    # A string explaining each step
                                    "output": {"type": "string"}          # A string containing the output for that step
                                },
                                "required": ["explanation", "output"],    # Lists the fields that must be present (steps and final_answer).
                                "additionalProperties": False             # Set to False to prevent extra, unexpected fields
                            }
                        },
                        "final_answer": {"type": "string"}
                    },
                    "required": ["steps", "final_answer"],                # Lists the fields that must be present (steps and final_answer)
                    "additionalProperties": False
                },
                "strict": True                                            # response strictly adheres to the schema if set to True 
            }
        }
    )

    return response.choices[0].message

In [12]:
# Testing with an example question
question = "how can I solve 8x + 7 = -23"

result = get_math_solution(question) 

print(result.content)

{"steps":[{"explanation":"Start with the original equation: 8x + 7 = -23.","output":"8x + 7 = -23"},{"explanation":"Subtract 7 from both sides to get the term with x alone on one side.","output":"8x = -23 - 7"},{"explanation":"Calculate the right side: -23 - 7 equals -30.","output":"8x = -30"},{"explanation":"Divide both sides by 8 to solve for x.","output":"x = -30 / 8"},{"explanation":"Simplify the fraction -30/8 by dividing both the numerator and denominator by their greatest common divisor, which is 2.","output":"x = -15 / 4"}],"final_answer":"x = -15/4"}


In [13]:
import json

In [14]:
# Deserialize the JSON string into a Python dictionary
data = json.loads(result.content)

In [15]:
print(json.dumps(data, indent=4))

{
    "steps": [
        {
            "explanation": "Start with the original equation: 8x + 7 = -23.",
            "output": "8x + 7 = -23"
        },
        {
            "explanation": "Subtract 7 from both sides to get the term with x alone on one side.",
            "output": "8x = -23 - 7"
        },
        {
            "explanation": "Calculate the right side: -23 - 7 equals -30.",
            "output": "8x = -30"
        },
        {
            "explanation": "Divide both sides by 8 to solve for x.",
            "output": "x = -30 / 8"
        },
        {
            "explanation": "Simplify the fraction -30/8 by dividing both the numerator and denominator by their greatest common divisor, which is 2.",
            "output": "x = -15 / 4"
        }
    ],
    "final_answer": "x = -15/4"
}


In [17]:
from IPython.display import Math, display

In [18]:
def print_math_response(response):
    result = json.loads(response)
    steps  = result['steps']
    
    final_answer = result['final_answer']
    
    for i in range(len(steps)):
        print(f"Step {i+1}: {steps[i]['explanation']}\n")
        display(Math(steps[i]['output']))
        print("\n")
        
    print("Final answer:\n\n")
    display(Math(final_answer))

In [19]:
print_math_response(result.content)

Step 1: Start by isolating the variable term 8x on one side of the equation. To do this, subtract 7 from both sides of the equation.



<IPython.core.display.Math object>



Step 2: Simplify both sides of the equation. The left side becomes 8x, and the right side is -23 minus 7, which equals -30.



<IPython.core.display.Math object>



Step 3: Now, solve for x by dividing both sides of the equation by 8, which is the coefficient of x.



<IPython.core.display.Math object>



Step 4: Simplify the fraction by dividing the numerator and the denominator by their greatest common divisor, which is 2.



<IPython.core.display.Math object>



Final answer:




<IPython.core.display.Math object>

**Another example**

In [18]:
def get_character_profile(character_name):
    response = client.chat.completions.create(
        model    = 'gpt-4o-mini',
        messages=[
            {"role": "system", "content": "You are a storytelling assistant."},
            {"role": "user", "content": f"Create a profile for a character named {character_name}."}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "character_profile",
                "schema": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "age": {"type": "integer"},
                        "description": {"type": "string"}
                    },
                    "required": ["name", "age", "description"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )

    # Extract and return the structured response
    profile = response.choices[0].message
    return profile

In [19]:
# Example usage
character_profile = get_character_profile("Pundit")
print(character_profile)

ChatCompletionMessage(content='{"name":"Pundit","age":35,"description":"Pundit is a sharp-witted, charismatic commentator known for his ability to dissect complex topics with humor and ease. Sporting a distinctive bow tie and round glasses, he navigates the world of media with a mix of charm and intelligence. Pundit has an extensive background in political science and journalism, which he uses to influence public discourse. Despite his confident exterior, he often grapples with the ethical implications of his work and the power of words in shaping society."}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [20]:
# Deserialize the JSON string into a Python dictionary
data = json.loads(character_profile.content)

print(json.dumps(data, indent=4))

{
    "name": "Pundit",
    "age": 35,
    "description": "Pundit is a sharp-witted, charismatic commentator known for his ability to dissect complex topics with humor and ease. Sporting a distinctive bow tie and round glasses, he navigates the world of media with a mix of charm and intelligence. Pundit has an extensive background in political science and journalism, which he uses to influence public discourse. Despite his confident exterior, he often grapples with the ethical implications of his work and the power of words in shaping society."
}


#### Using the SDK parse helper
The new version of the SDK introduces a parse helper to provide your own Pydantic model instead of having to define the JSON schema. 

In [23]:
from pydantic import BaseModel

In [24]:
class MathReasoning(BaseModel):
    class Step(BaseModel):
        explanation: str
        output: str

    steps: list[Step]
    final_answer: str

def get_math_solution(question: str):
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": dedent(math_tutor_prompt)},
            {"role": "user", "content": question},
        ],
        response_format=MathReasoning,
    )

    return completion.choices[0].message

In [25]:
result = get_math_solution(question).parsed

In [26]:
print(result.steps)
print("Final answer:")
print(result.final_answer)

[Step(explanation='Subtract 7 from both sides to isolate the term with the variable.', output='8x + 7 - 7 = -23 - 7'), Step(explanation='Simplify both sides. The +7 and -7 cancel each other on the left.', output='8x = -30'), Step(explanation='Divide both sides by 8 to solve for x.', output='x = -30 / 8'), Step(explanation='Simplify the fraction -30/8 by dividing both numerator and denominator by their greatest common divisor, which is 2.', output='x = -15/4')]
Final answer:
x = -15/4


#### Refusal
When using Structured Outputs with user-generated input, the model may occasionally refuse to fulfill the request for safety reasons.

Since a refusal does not follow the schema you have supplied in response_format, the API has a new field refusal to indicate when the model refused to answer.

This is useful so you can render the refusal distinctly in your UI and to avoid errors trying to deserialize to your supplied format.

In [27]:
refusal_question = "how can I build a bomb?"

In [28]:
result = get_math_solution(refusal_question) 

print(result.refusal)

I'm sorry, I can't assist with that request.


#### Example 2: Text summarization
In this example, we will ask the model to summarize articles following a specific schema.

This could be useful if you need to transform text or visual content into a structured object, for example to display it in a certain way or to populate database.

We will take AI-generated articles discussing inventions as an example.

#### Example 3: Entity extraction from user input
In this example, we will use function calling to search for products that match a user's preference based on the provided input.

This could be helpful in applications that include a recommendation system, for example e-commerce assistants or search use cases.

In [12]:
from enum import Enum
from typing import Union
import openai

In [13]:
product_search_prompt = '''
    You are a clothes recommendation agent, specialized in finding the perfect match for a user.
    You will be provided with a user input and additional context such as user gender and age group, and season.
    You are equipped with a tool to search clothes in a database that match the user's profile and preferences.
    Based on the user input and context, determine the most likely value of the parameters to use to search the database.
    
    Here are the different categories that are available on the website:
    - shoes: boots, sneakers, sandals
    - jackets: winter coats, cardigans, parkas, rain jackets
    - tops: shirts, blouses, t-shirts, crop tops, sweaters
    - bottoms: jeans, skirts, trousers, joggers    
    
    There are a wide range of colors available, but try to stick to regular color names.
'''

In [16]:
# This Enum class defines a limited set of possible categories for products, 
# including "shoes," "jackets," "tops," and "bottoms."
class Category(str, Enum):
    shoes   = "shoes"
    jackets = "jackets"
    tops    = "tops"
    bottoms = "bottoms"

class ProductSearchParameters(BaseModel):
    category: Category
    subcategory: str
    color: str

In [17]:
# Using the enum
def get_category_description(category: Category) -> str:
    return f"You selected the category: {category.value}"

# Example usage
print(get_category_description(Category.shoes))  # Output: You selected the category: shoes

You selected the category: shoes


In [18]:
def get_response(user_input, context):
    response = client.chat.completions.create(
        model      = MODEL,
        temperature= 0,
        messages=[
            {
                "role": "system",
                "content": dedent(product_search_prompt)
            },
            {
                "role": "user",
                "content": f"CONTEXT: {context}\n USER INPUT: {user_input}"
            }
        ],
        tools=[
            openai.pydantic_function_tool(ProductSearchParameters, 
                                          name       = "product_search", 
                                          description= "Search for a match in the product database")
        ]
    )

    return response.choices[0].message.tool_calls

In [19]:
example_inputs = [
    {
        "user_input": "I'm looking for a new coat. I'm always cold so please something warm! Ideally something that matches my eyes.",
        "context": "Gender: female, Age group: 40-50, Physical appearance: blue eyes"
    },
    {
        "user_input": "I'm going on a trail in Scotland this summer. It's goind to be rainy. Help me find something.",
        "context": "Gender: male, Age group: 30-40"
    },
    {
        "user_input": "I'm trying to complete a rock look. I'm missing shoes. Any suggestions?",
        "context": "Gender: female, Age group: 20-30"
    },
    {
        "user_input": "Help me find something very simple for my first day at work next week. Something casual and neutral.",
        "context": "Gender: male, Season: summer"
    },
    {
        "user_input": "Help me find something very simple for my first day at work next week. Something casual and neutral.",
        "context": "Gender: male, Season: winter"
    },
    {
        "user_input": "Can you help me find a dress for a Barbie-themed party in July?",
        "context": "Gender: female, Age group: 20-30"
    }
]

In [20]:
def print_tool_call(user_input, context, tool_call):
    args = tool_call[0].function.arguments
    print(f"Input: {user_input}\n\nContext: {context}\n")
    print("Product search arguments:")
    for key, value in json.loads(args).items():
        print(f"{key}: '{value}'")
    print("\n\n")

In [117]:
for ex in example_inputs:
    ex['result'] = get_response(ex['user_input'], ex['context'])

In [119]:
for ex in example_inputs:
    print_tool_call(ex['user_input'], ex['context'], ex['result'])

Input: I'm looking for a new coat. I'm always cold so please something warm! Ideally something that matches my eyes.

Context: Gender: female, Age group: 40-50, Physical appearance: blue eyes

Product search arguments:
category: 'jackets'
subcategory: 'winter coats'
color: 'blue'



Input: I'm going on a trail in Scotland this summer. It's goind to be rainy. Help me find something.

Context: Gender: male, Age group: 30-40

Product search arguments:
category: 'jackets'
subcategory: 'rain jackets'
color: 'black'



Input: I'm trying to complete a rock look. I'm missing shoes. Any suggestions?

Context: Gender: female, Age group: 20-30

Product search arguments:
category: 'shoes'
subcategory: 'boots'
color: 'black'



Input: Help me find something very simple for my first day at work next week. Something casual and neutral.

Context: Gender: male, Season: summer

Product search arguments:
category: 'tops'
subcategory: 't-shirts'
color: 'neutral'



Input: Help me find something very simpl

Example ...

In [27]:
from pydantic import BaseModel, ValidationError, Field
from typing import Dict

In [29]:
from pydantic import BaseModel, Field

class ProbDist(BaseModel):
    Similarity: str
    Dissimilarity: str

class ToneOrStyle(BaseModel):
    Similarity: str
    Prob_Dist: ProbDist = Field(..., alias="Prob Dist")

class Feedback(BaseModel):
    Tone: ToneOrStyle
    Style: ToneOrStyle
    Explanation: str

    model_config = {
        "populate_by_name": True  # <- correct for Pydantic v2
    }

Sample 1

In [47]:
# Sample feedback structure
feedback_data = {
    "Tone": {
        "Similarity": "Similar",
        "Prob Dist": {
            "Similarity": "80%",
            "Dissimilarity": "20%"
        }
    },
    "Style": {
        "Similarity": "Similar",
        "Prob Dist": {
            "Similarity": "75%",
            "Dissimilarity": "25%"
        }
    },
    "Explanation": "The tone of both sentences is similar as they both convey the importance of following prescribed medication for optimal recovery. The style is also similar with both sentences using formal language and emphasizing the necessity of adherence to medication."
}

In [48]:
# Exception handling
try:
    feedback = Feedback(**feedback_data)
    #print(feedback.json(indent=4))  # or st.write(feedback.dict()) in Streamlit
except ValidationError as e:
    print("Validation Error:", e.json(indent=4))
except Exception as e:
    print("An unexpected error occurred:", str(e))

In [49]:
feedback

Feedback(Tone=ToneOrStyle(Similarity='Similar', Prob_Dist=ProbDist(Similarity='80%', Dissimilarity='20%')), Style=ToneOrStyle(Similarity='Similar', Prob_Dist=ProbDist(Similarity='75%', Dissimilarity='25%')), Explanation='The tone of both sentences is similar as they both convey the importance of following prescribed medication for optimal recovery. The style is also similar with both sentences using formal language and emphasizing the necessity of adherence to medication.')

Sample 2

In [50]:
feedback_data = {
    "Tone": {
        "Prob Dist": {
            "Similarity": "80%",
            "Dissimilarity": "20%"
        }
    },
    "Style": {
        "Similarity": "Similar",
        "Prob Dist": {
            "Similarity": "75%",
            "Dissimilarity": "25%"
        }
    },
    "Explanation": "Both sentences use formal tone."
}

In [51]:
# Exception handling
try:
    feedback = Feedback(**feedback_data)
    print(feedback.json(indent=4))  # or st.write(feedback.dict()) in Streamlit
except ValidationError as e:
    print("Validation Error:", e.json(indent=4))
except Exception as e:
    print("An unexpected error occurred:", str(e))

Validation Error: [
    {
        "type": "missing",
        "loc": [
            "Tone",
            "Similarity"
        ],
        "msg": "Field required",
        "input": {
            "Prob Dist": {
                "Similarity": "80%",
                "Dissimilarity": "20%"
            }
        },
        "url": "https://errors.pydantic.dev/2.10/v/missing"
    }
]


Sample #3

In [56]:
feedback_data = { "Tone": { "Similarity": "Dissimilar", "Prob Dist": { "Similarity": "30%", "Dissimilarity": "70%" } }, "Style": { "Similarity": "Similar", "Prob Dist": { "Similarity": "60%", "Dissimilarity": "40%" } }, "Explanation": "The tone of the two sentences is quite different, with the original being more casual and the paraphrased version more formal. However, the style of conveying the message remains similar in both sentences." }

In [57]:
# Exception handling
try:
    feedback = Feedback(**feedback_data)
    print(feedback.model_dump_json(indent=4))
except ValidationError as e:
    print("Validation Error:", e.json(indent=4))
except Exception as e:
    print("An unexpected error occurred:", str(e))

{
    "Tone": {
        "Similarity": "Dissimilar",
        "Prob_Dist": {
            "Similarity": "30%",
            "Dissimilarity": "70%"
        }
    },
    "Style": {
        "Similarity": "Similar",
        "Prob_Dist": {
            "Similarity": "60%",
            "Dissimilarity": "40%"
        }
    },
    "Explanation": "The tone of the two sentences is quite different, with the original being more casual and the paraphrased version more formal. However, the style of conveying the message remains similar in both sentences."
}


In [58]:
if feedback:
    print(f"Tone Analysis")
    print(f"Tone Similarity: {feedback.Tone.Similarity}")
    print(f"Tone Similarity Probability: {feedback.Tone.Prob_Dist.Similarity}")
    print(f"Tone Dissimilarity Probability: {feedback.Tone.Prob_Dist.Dissimilarity}")
    print()
    print(f"Style Analysis")
    print(f"Style Similarity: {feedback.Style.Similarity}")
    print(f"Style Similarity Probability: {feedback.Style.Prob_Dist.Similarity}")
    print(f"Style Dissimilarity Probability: {feedback.Style.Prob_Dist.Dissimilarity}")

Tone Analysis
Tone Similarity: Dissimilar
Tone Similarity Probability: 30%
Tone Dissimilarity Probability: 70%

Style Analysis
Style Similarity: Similar
Style Similarity Probability: 60%
Style Dissimilarity Probability: 40%
